In [ ]:
# # Install all required libraries in one shot
# !pip install -q torch torchvision transformers peft bitsandbytes accelerate \
#              nltk rouge-score evaluate bert_score qwen-vl-utils
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

## 1 · Imports & Config

In [ ]:
import os, json, random, gc, shutil, traceback, zipfile, warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
import matplotlib.pyplot as plt

from transformers import (
    AutoModel, AutoTokenizer, AutoProcessor,
    Qwen2VLForConditionalGeneration,
    TrainingArguments, Trainer, BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import evaluate

warnings.filterwarnings('ignore')
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)

## 2 · Dataset Paths

In [ ]:
KAGGLE_BASE = '/kaggle/input/datasets/windyy261203/openvivqa'
LOCAL_BASE  = 'data/openvivqa'
BASE_DIR    = KAGGLE_BASE if os.path.exists(KAGGLE_BASE) else LOCAL_BASE
CKPT_DIR    = '/kaggle/working' if os.path.exists('/kaggle/working') else './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Dataset:', BASE_DIR)
print('Checkpoints:', CKPT_DIR)

## 3 · Load OpenViVQA

In [ ]:
def build_img_map(img_dir):
    img_map = {}
    for root, _, files in os.walk(img_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_map[f] = os.path.join(root, f)
    return img_map

def parse_annotations(json_path, img_dir):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    img_map = build_img_map(img_dir)
    parsed, missing = [], 0

    if isinstance(data, list):  # Format A
        print(f'  Format A - {len(data)} records')
        for item in data:
            fname = os.path.basename(item.get('image', ''))
            if fname in img_map:
                parsed.append({'image': img_map[fname],
                               'question': item['question'],
                               'answer': item['answer']})
            else:
                missing += 1
    elif isinstance(data, dict) and 'annotations' in data:  # Format B
        images_dict = data.get('images', {})
        print(f'  Format B - {len(data["annotations"])} annotations')
        for ann in data['annotations'].values():
            fname = os.path.basename(images_dict.get(str(ann.get('image_id', '')), ''))
            if fname and fname in img_map:
                parsed.append({'image': img_map[fname],
                               'question': ann['question'],
                               'answer': ann['answer']})
            else:
                missing += 1
    else:
        raise ValueError(f'Unknown JSON format: {json_path}')

    print(f'  => {len(parsed)} valid QA pairs, {missing} missing images')
    return parsed

train_data, val_data, test_data = [], [], []
for split, (ann_file, img_folder) in {
    'train': ('training-annotations.json', 'training-images'),
    'val':   ('dev-annotations.json',      'dev-images'),
    'test':  ('test-annotations.json',     'test-images'),
}.items():
    json_path = os.path.join(BASE_DIR, ann_file)
    img_dir   = os.path.join(BASE_DIR, img_folder)
    print(f'\n--- {split.upper()} ---')
    if not os.path.exists(json_path):
        print(f'  NOT FOUND: {json_path}')
        continue
    parsed = parse_annotations(json_path, img_dir)
    if split == 'train':  train_data = parsed
    elif split == 'val':  val_data   = parsed
    else:                 test_data  = parsed

print(f'\nTotal - Train:{len(train_data)} Val:{len(val_data)} Test:{len(test_data)}')

In [ ]:
from functools import partial

class Qwen2VLDataset(Dataset):
    """
    Formats each sample as a user→assistant conversation.
    Labels mask the prompt tokens with -100 so the model only
    learns to predict the answer portion.
    """
    def __init__(self, data_list, processor):
        self.data = data_list
        self.processor = processor
        self.tokenizer = processor.tokenizer

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        try:    image = Image.open(item['image']).convert('RGB')
        except: image = Image.new('RGB', (224, 224))

        # Build prompt-only text (to compute its token length)
        prompt_msgs = [{'role': 'user', 'content': [
            {'type': 'image'}, {'type': 'text', 'text': item['question']}
        ]}]
        prompt_text = self.processor.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True
        )

        # Full conversation (prompt + answer)
        full_msgs = prompt_msgs + [{
            'role': 'assistant',
            'content': [{'type': 'text', 'text': item['answer']}]
        }]
        full_text = self.processor.apply_chat_template(
            full_msgs, tokenize=False, add_generation_prompt=False
        )

        inputs = self.processor(
            text=[full_text], images=[image],
            padding=False, return_tensors='pt'
        )

        input_ids = inputs.input_ids[0]
        labels    = input_ids.clone()

        # Mask prompt tokens in labels
        prompt_len = len(
            self.tokenizer(
                prompt_text,
                add_special_tokens=False
            )["input_ids"]
        )
        
        labels[:prompt_len] = -100

        return {
            'input_ids':      input_ids,
            'attention_mask': inputs.attention_mask[0],
            'labels':         labels,
            'pixel_values':   inputs.pixel_values,
            'image_grid_thw': inputs.image_grid_thw,
        }

def qwen_data_collator(features,pad_token_id):
    """Pad sequences; pixel_values/image_grid_thw are concatenated."""
    pad_id = -100  # labels use -100; input_ids will be corrected by attention_mask
    return {
        'input_ids': torch.nn.utils.rnn.pad_sequence(
            [f['input_ids'] for f in features], batch_first=True, padding_value=pad_token_id
        ),
        'attention_mask': torch.nn.utils.rnn.pad_sequence(
            [f['attention_mask'] for f in features], batch_first=True, padding_value=0
        ),
        'labels': torch.nn.utils.rnn.pad_sequence(
            [f['labels'] for f in features], batch_first=True, padding_value=-100
        ),
        'pixel_values':   torch.cat([f['pixel_values'] for f in features]),
        'image_grid_thw': torch.cat([f['image_grid_thw'] for f in features]),
    }

def run_finetune_qwen():
    model_id  = 'Qwen/Qwen2-VL-2B-Instruct'
    final_dir = './qwen2vl_finetuned_final'

    try:
        processor = AutoProcessor.from_pretrained(model_id, max_pixels=150000)

        pad_token_id = processor.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = processor.tokenizer.eos_token_id
        
        data_collator = partial(
            qwen_data_collator,
            pad_token_id=pad_token_id
        )

        train_ds = Qwen2VLDataset(train_data, processor)
        val_ds   = Qwen2VLDataset(val_data,   processor)

        # 4-bit quantization - compatible with single-device mapping (no DataParallel)
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
        # device_map='auto' distributes across available GPUs automatically
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_cfg,
            torch_dtype=torch.float16,
            #device_map='auto',
        )

        # Required before applying LoRA on a quantized model
        model = prepare_model_for_kbit_training(model)

        model.gradient_checkpointing_enable()
        model.config.use_cache = False

        lora_cfg = LoraConfig(
            r=16, lora_alpha=32,
            target_modules=['q_proj','k_proj','v_proj','o_proj',
                            'gate_proj','up_proj','down_proj'],
            task_type='CAUSAL_LM',
            lora_dropout=0.05,
        )
        model = get_peft_model(model, lora_cfg)
        model.print_trainable_parameters()

        training_args = TrainingArguments(
            output_dir="./qwen2vl_finetuned",

            # Batching
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=16,
        
            # Training length
            max_steps=1500,

            # Optimizer
            learning_rate=2e-4,
            warmup_steps=50,
            lr_scheduler_type="cosine",
            weight_decay=0.01,
        
            # Precision
            fp16=True,
        
            # Logging
            logging_strategy="steps",
            logging_steps=20,
        
            eval_strategy="steps",
            eval_steps=200,
            eval_accumulation_steps=1,
        
            save_strategy="steps",
            save_steps=200,
            save_total_limit=2,
        
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        

            remove_unused_columns=False,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
            
            report_to="none",
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=data_collator,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

        trainer.train()

        trainer.save_model(final_dir)
        processor.save_pretrained(final_dir)
        print(f'Fine-tuned model saved to {final_dir}')

    except Exception:
        traceback.print_exc()

run_finetune_qwen()